<table width="100%" bgcolor="#f4f5f6" style="width: 100%; background-color: #f4f5f6; border-left: 6px solid #ff9900; border-radius: 8px; border-collapse: collapse;">
  <tr>
    <td width="110" align="center" valign="middle" style="padding: 15px;">
      <img src="https://github.com/sansalamanca/shining-path_DP700/raw/main/data/logo%20libro.jpg" width="85" style="border-radius: 8px; display: block;">
    </td>
    <td valign="middle" style="padding: 15px 20px 15px 5px; font-family: Arial, sans-serif;">
      <p style="margin: 0 0 4px 0; font-size: 13px; color: #555555;">Excerpt from the Amazon book</p>
      <h2 style="margin: 4px 0 10px 0; font-size: 20px; font-weight: 900; color: #111111; border: none; padding: 0;">THE SHINING PATH TO DATA ENGINEER CERTIFICATE</h2>
      <p style="margin: 0; font-size: 14px; font-weight: bold;">
        <a href="https://www.amazon.com" target="_blank" style="color: #b12704; text-decoration: underline;">🛒 Buy on Amazon &rarr;</a>
      </p>
    </td>
  </tr>
</table>

## **Delta Lake (Ideal for transactional databases, advanced analytics, and version control)**
Delta Lake adds ACID transactions on top of Parquet files.

In [7]:
#1. Install PySpark and Delta Lake (silencing Colab internal package warnings)
!pip install -q --no-warn-conflicts pyspark==3.5.3 delta-spark==3.2.1

# 2. clone folders from  GitHub
!rm -rf shining-path_DP700
!git clone -q https://github.com/sansalamanca/shining-path_DP700.git

# 3. Initialize the SparkSession configured for Delta Lake
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Fabric_Lakehouse_Simulation") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("SparkSession with Delta Lake ready!")

SparkSession with Delta Lake ready!


In [8]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Define schema of products.csv
schema = StructType([
    StructField("ProductID", IntegerType(), True),
    StructField("ProductName", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("ListPrice", DoubleType(), True)
])

# 1. Read CSV from Files/   folder
df_products = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load("shining-path_DP700/Lakehouse/Files/products.csv")

# 2. Save Delta table in Tables/
delta_table_path = "shining-path_DP700/Lakehouse/Tables/products_delta"
df_products.write.format("delta").mode("overwrite").save(delta_table_path)

print("The Delta table has been successfully saved in Lakehouse/Tables/products_delta!")

The Delta table has been successfully saved in Lakehouse/Tables/products_delta!


In [9]:
# 1. Read Delta table from Tables/
path_delta = "shining-path_DP700/Lakehouse/Tables/products_delta"
df_delta_red = spark.read.format("delta").load(path_delta)

# 2. Display DF and verify  its schema
df_delta_red.show(10, truncate=False)
df_delta_red.printSchema()

+---------+-----------------------+--------------+---------+
|ProductID|ProductName            |Category      |ListPrice|
+---------+-----------------------+--------------+---------+
|771      |Mountain-100 Silver, 38|Mountain Bikes|3399.99  |
|772      |Mountain-100 Silver, 42|Mountain Bikes|3399.99  |
|773      |Mountain-100 Silver, 44|Mountain Bikes|3399.99  |
|774      |Mountain-100 Silver, 48|Mountain Bikes|3399.99  |
|775      |Mountain-100 Black, 38 |Mountain Bikes|3374.99  |
|776      |Mountain-100 Black, 42 |Mountain Bikes|3374.99  |
|777      |Mountain-100 Black, 44 |Mountain Bikes|3374.99  |
|778      |Mountain-100 Black, 48 |Mountain Bikes|3374.99  |
|779      |Mountain-200 Silver, 38|Mountain Bikes|2319.99  |
|780      |Mountain-200 Silver, 42|Mountain Bikes|2319.99  |
+---------+-----------------------+--------------+---------+
only showing top 10 rows

root
 |-- ProductID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullabl